# Predictive Maintenance - Model Prediction Demonstration
This notebook demonstrates how we load the trained machine learning model and use it to predict equipment failure based on IoT sensor data. This is an ideal demonstration for an interview context.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

## 1. Load the Model and Scaler
We load the best model and the `StandardScaler` that were saved during the training phase.

In [ ]:
# Load the saved model and scaler
model = joblib.load('models/best_model.pkl')
scaler = joblib.load('models/scaler.pkl')

print(f"Model loaded successfully: {type(model).__name__}")

## 2. Prepare Sample Data
Let's create some sample sensor readings. In a real-world scenario, this data would stream from IoT devices or via the Streamlit dashboard.

The dataset features used are: `temperature`, `vibration`, `pressure`, `current`, `voltage`, `rpm`, `humidity`, and `load`.

In [ ]:
# Define the features expected by the model
features = ['temperature', 'vibration', 'pressure', 'current', 'voltage', 'rpm', 'humidity', 'load']

# Create hypothetical sensor data (one normal case, one anomalous case)
sample_data = pd.DataFrame([
    [65.0, 2.5, 100.0, 15.0, 220.0, 1500, 45.0, 80.0],  # Normal machine state
    [95.0, 8.5, 130.0, 25.0, 210.0, 3000, 60.0, 95.0]   # High temp, high vibration -> likely failure
], columns=features)

display(sample_data)

## 3. Preprocess and Predict
We scale the input data using the loaded scaler, then generate predictions and the probability of failure.

In [ ]:
# Scale the sample features
sample_scaled = scaler.transform(sample_data)

# Generate predictions
predictions = model.predict(sample_scaled)

# Get probabilities (assuming the model supports predict_proba)
if hasattr(model, 'predict_proba'):
    probabilities = model.predict_proba(sample_scaled)[:, 1] # Probability of failure (class 1)
else:
    probabilities = ['N/A'] * len(predictions)

# Append results to our DataFrame for a clear view
sample_data['Failure_Prediction'] = predictions
sample_data['Failure_Probability'] = probabilities

# Add a human-readable label
sample_data['Status'] = sample_data['Failure_Prediction'].apply(lambda x: 'Needs Maintenance ⚠️' if x == 1 else 'Healthy ✅')

# Display the final predictive summary
display(sample_data[['Status', 'Failure_Probability', 'temperature', 'vibration', 'rpm']])

## 4. Bulk Prediction from CSV (Optional Demonstration)
Below is how we would apply this to a batch of records, exactly like the "Bulk Prediction" feature of our app.

In [ ]:
import os
data_path = 'data/sensor_data.csv'

if os.path.exists(data_path):
    # Load top 5 rows from the actual dataset
    real_data = pd.read_csv(data_path).head(5)
    
    # Extract features
    X_real = real_data[features]
    
    # Process and predict
    X_real_scaled = scaler.transform(X_real)
    real_preds = model.predict(X_real_scaled)
    
    # Show results
    real_data['Predicted_Status'] = ['Failure ⚠️' if p == 1 else 'Healthy ✅' for p in real_preds]
    display(real_data[['machine_id', 'Predicted_Status', 'failure_status']])
else:
    print("Dataset not found. Please run the data generator first.")